# Fig. S4, D and E: developmental APA landscape (proportion-based)

Supports Fig. 2A and 2B. S3a per-cell-type density along an unbiased differentiation
axis (principal curve, Hastie-Stuetzle 1989, PCA-initialised, no start/end-cluster bias) computed
on the 3'UTR-proportion UMAP, per lineage: 3'UTR isoform proportions alone order cells from
progenitor -> mature. S3b the delta-psi pseudotime volcano (the effect-size view; the main
volcano, Fig. 2B, uses rho_W). Sub-panels under `Fig2/supp_3/<panel>/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde, spearmanr
from scipy.interpolate import UnivariateSpline
from sklearn.neighbors import NearestNeighbors

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()
FIG = "Fig2/supp_3"

d = np.load(
    S.OUT_ROOT / "Fig2" / "fig2b_proportions_umap" / "_prop_umap_axis.npz",
    allow_pickle=True,
)
um, ct, lin = d["umap"], d["celltype"].astype(str), d["lineage"].astype(str)
print("loaded proportion-UMAP cache:", um.shape)

In [ ]:
# principal curve (Hastie & Stuetzle 1989), PCA-initialised -> unbiased to start/end clusters
def _spl(t, X, s, col):
    tu, inv = np.unique(t, return_inverse=True)
    ys = np.zeros(len(tu))
    np.add.at(ys, inv, X[:, col])
    ym = ys / np.bincount(inv)
    k = min(3, len(tu) - 1)
    return (
        (lambda x: np.full_like(x, ym.mean()))
        if k < 1
        else UnivariateSpline(tu, ym, k=k, s=s, ext=3)
    )


def principal_curve(X, n_iter=8, smoothing=5, grid=300, max_fit=6000, seed=0):
    # Fit the smoothing spline on a seeded subsample (identical curve shape, but the
    # UnivariateSpline fit on all ~47k points was prohibitively slow); every cell is still
    # projected onto the final curve, so the per-cell differentiation axis is unchanged.
    n = len(X)
    s = min(n, max_fit) / smoothing
    t = PCA(1).fit_transform(X).ravel()
    t -= t.min()
    rng = np.random.default_rng(seed)
    fit_idx = rng.choice(n, size=min(n, max_fit), replace=False) if n > max_fit else np.arange(n)
    for _ in range(n_iter):
        tf, Xf = t[fit_idx], X[fit_idx]
        o = np.argsort(tf)
        ts, Xs = tf[o], Xf[o]
        g = np.linspace(ts.min(), ts.max(), grid)  # Fixed coarse grid (speed)
        cp = np.column_stack([_spl(ts, Xs, s, 0)(g), _spl(ts, Xs, s, 1)(g)])
        if not np.all(np.isfinite(cp)):
            break
        arc = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(cp, axis=0), axis=1))])
        _, idx = NearestNeighbors(n_neighbors=1).fit(cp).kneighbors(X)
        t = arc[idx.ravel()]
    return t

In [ ]:
# S3a - per-cell-type differentiation density (principal-curve axis), per lineage
LINS = [
    ("Excitatory neurons", S.ORGANOID_EXC_ORDER, S.paper_palette_lineage["excitatory"]),
    ("Inhibitory neurons", S.ORGANOID_INH_ORDER, S.paper_palette_lineage["inhibitory"]),
]
fig, axes = plt.subplots(
    1,
    2,
    figsize=(S.FULL_W_PX / 300, S.mm_to_px(95) / 300),
    gridspec_kw={"width_ratios": [1, 1]},
)
for ax, (L, order, lcol) in zip(axes, LINS):
    lm = lin == L
    sub_um = um[lm]
    sub_ct = ct[lm]
    cts = [c for c in order if (sub_ct == c).any()]
    axis = principal_curve(sub_um.astype(float))
    means = [axis[sub_ct == c].mean() for c in cts]
    if spearmanr(range(len(cts)), means)[0] < 0:
        axis = -axis
    axis = (axis - axis.min()) / (axis.max() - axis.min() + 1e-9)
    xr = np.linspace(-0.08, 1.08, 400)
    rh = 1.0
    for i, c in enumerate(cts):
        cd = axis[sub_ct == c]
        if len(cd) < 5:
            continue
        kde = gaussian_kde(cd, bw_method=0.35)
        dens = kde(xr)
        dens = dens / dens.max() * rh * 0.92
        # Progenitors at top, differentiated at bottom (flip row placement;
        # the x differentiation axis orientation is unchanged)
        y0 = (len(cts) - 1 - i) * rh
        col = S.paper_palette_organoid_celltypes.get(c, "#888888")
        ax.fill_between(xr, y0, y0 + dens, color=col, alpha=0.8, zorder=i + 2, linewidth=0)
        ax.plot(xr, y0 + dens, color=col, lw=0.5, zorder=i + 3)
        ax.axhline(y0, color="0.9", lw=0.3, zorder=1)
        med = np.median(cd)
        mh = (kde([med])[0] / kde(xr).max()) * rh * 0.92
        ax.plot(
            [med, med],
            [y0, y0 + mh],
            color="white",
            lw=0.8,
            ls="--",
            zorder=i + 4,
            alpha=0.9,
        )
        ax.text(
            -0.12,
            y0 + rh * 0.35,
            c,
            fontsize=S.FONT_TICK - 0.5,
            va="center",
            ha="right",
            color=col,
            fontweight="bold",
        )
    ax.annotate(
        "",
        xy=(1.08, -0.7),
        xytext=(-0.08, -0.7),
        arrowprops=dict(arrowstyle="-|>", color="dimgray", lw=0.9, mutation_scale=7),
    )
    ax.text(
        0.5,
        -1,
        "differentiation (proportion principal curve)",
        fontsize=S.FONT_TICK,
        ha="center",
        color="dimgray",
        style="italic",
    )
    ax.set_title(L, fontsize=S.FONT_LABEL, color=lcol, fontweight="bold", pad=3)
    ax.set_yticks([])
    ax.set_xticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
fig.subplots_adjust(left=0.10, right=0.98, top=0.93, bottom=0.07, wspace=0.30)
S.save_and_show(fig, "fig_s3a_differentiation_density", figure=FIG)

In [ ]:
# S3b - Delta-psi pseudotime volcano (the effect-size view; the main fig2c volcano uses rho_W)
import pandas as pd
from matplotlib.lines import Line2D

try:
    from adjustText import adjust_text

    HAS_AT = True
except ImportError:
    HAS_AT = False
KB = f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling"
LINS = [("excitatory", "Excitatory neurons"), ("inhibitory", "Inhibitory neurons")]
PADJ_THR = 0.05
ASD_STAR = "#e08214"
comp = {
    lin_: pd.read_csv(f"{KB}/composite_ranking/composite_ranking_optimized2_{lin_}.csv")
    for lin_, _ in LINS
}
asd = pd.read_csv(f"{KB}/asd_results/asd_significant_in_knoblich.csv")
for lin_, _ in LINS:
    d = comp[lin_]
    d["nl"] = -np.log10(d["padj"].clip(lower=1e-300))
    cap = d.loc[np.isfinite(d["nl"]), "nl"].quantile(0.999)
    d["nl"] = d["nl"].clip(upper=cap * 1.05)
    d["sig"] = np.where(
        (d["padj"] < PADJ_THR) & (d["delta_psi"] < 0),
        "short",
        np.where((d["padj"] < PADJ_THR) & (d["delta_psi"] > 0), "long", "NS"),
    )
fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, 780 / 300))
for ax, (lin_, linlabel) in zip(axes, LINS):
    d = comp[lin_]
    cs = d["composite_score_abs"].values
    size = 1 + 12 * (cs - np.nanmin(cs)) / (np.nanmax(cs) - np.nanmin(cs) + 1e-9)
    for cls, col, z in [
        ("NS", S.COL_NS, 1),
        ("short", S.COL_SHORT, 2),
        ("long", S.COL_LONG, 2),
    ]:
        m = (d["sig"] == cls).values
        ax.scatter(
            d.loc[m, "delta_psi"],
            d.loc[m, "nl"],
            c=col,
            s=size[m],
            alpha=0.35 if cls == "NS" else 0.7,
            edgecolors="none",
            rasterized=True,
            zorder=z,
        )
    ax.axhline(-np.log10(PADJ_THR), color="gray", ls="--", lw=0.6, zorder=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.18)
    a = asd[asd["lineage"] == lin_]
    if len(a):
        am = d["gene_name"].isin(a["gene_name"]).values
        ax.scatter(
            d.loc[am, "delta_psi"],
            d.loc[am, "nl"],
            s=18,
            marker="*",
            facecolors="none",
            edgecolors=ASD_STAR,
            linewidths=0.5,
            zorder=5,
        )
    top = d[d["sig"] != "NS"].nlargest(6, "composite_score_abs")
    texts = [
        ax.text(
            r["delta_psi"],
            r["nl"],
            r["gene_name"],
            fontsize=S.FONT_ANNOT - 0.5,
            fontstyle="italic",
            zorder=10,
        )
        for _, r in top.iterrows()
    ]
    if HAS_AT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.4))
    ns = int((d["sig"] == "short").sum())
    nl_ = int((d["sig"] == "long").sum())
    ax.text(
        0.02,
        0.97,
        f"n = {ns:,}",
        transform=ax.transAxes,
        color=S.COL_SHORT,
        fontsize=S.FONT_ANNOT,
        fontweight="bold",
        va="top",
        ha="left",
    )
    ax.text(
        0.98,
        0.97,
        f"n = {nl_:,}",
        transform=ax.transAxes,
        color=S.COL_LONG,
        fontsize=S.FONT_ANNOT,
        fontweight="bold",
        va="top",
        ha="right",
    )
    ax.set_title(
        linlabel,
        fontsize=S.FONT_LABEL,
        color=S.paper_palette_lineage[lin_],
        fontweight="bold",
        pad=3,
    )
    ax.set_xlabel(r"$\Delta\psi$ (pseudotime)", fontsize=S.FONT_LABEL)
    ax.set_ylabel(r"$-\log_{10}(p_{\mathrm{adj}})$", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
leg = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.COL_SHORT,
        markersize=4,
        label="shortening",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.COL_LONG,
        markersize=4,
        label="lengthening",
    ),
    Line2D(
        [0],
        [0],
        marker="*",
        color="w",
        markerfacecolor="none",
        markeredgecolor=ASD_STAR,
        markersize=6,
        label="ASD-risk gene",
    ),
]
axes[0].legend(
    handles=leg,
    fontsize=S.FONT_LEGEND,
    frameon=False,
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(1.05, 1.15),
    handletextpad=0.2,
    columnspacing=0.8,
)
fig.subplots_adjust(left=0.07, right=0.97, top=0.84, bottom=0.13, wspace=0.18)
S.save_and_show(fig, "fig_s3b_pseudotime_dpsi_volcano", figure=FIG)

## S3a + S3b size variants (quarter height; full/half width)


In [ ]:
# --- S3a (differentiation density) + S3b (dpsi volcano) size variants ---
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D

_d3 = np.load(
    S.OUT_ROOT / "Fig2" / "fig2b_proportions_umap" / "_prop_umap_axis.npz",
    allow_pickle=True,
)
_um3, _ct3, _lin3 = (
    _d3["umap"].astype(float),
    _d3["celltype"].astype(str),
    _d3["lineage"].astype(str),
)
_LINS_S3A = [
    ("Excitatory neurons", S.ORGANOID_EXC_ORDER, S.paper_palette_lineage["excitatory"]),
    ("Inhibitory neurons", S.ORGANOID_INH_ORDER, S.paper_palette_lineage["inhibitory"]),
]


def _build_s3a(figsize, half):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    lblf = S.FONT_TICK - 1 if half else S.FONT_TICK - 0.5
    for ax, (L, order, lcol) in zip(axes, _LINS_S3A):
        lm = _lin3 == L
        sub_um = _um3[lm]
        sub_ct = _ct3[lm]
        cts = [c for c in order if (sub_ct == c).any()]
        axis = principal_curve(sub_um.astype(float))
        means = [axis[sub_ct == c].mean() for c in cts]
        if spearmanr(range(len(cts)), means)[0] < 0:
            axis = -axis
        axis = (axis - axis.min()) / (axis.max() - axis.min() + 1e-9)
        xr = np.linspace(-0.08, 1.08, 400)
        rh = 1.0
        for i, c in enumerate(cts):
            cd = axis[sub_ct == c]
            if len(cd) < 5:
                continue
            kde = gaussian_kde(cd, bw_method=0.35)
            dens = kde(xr)
            dens = dens / dens.max() * rh * 0.92
            y0 = (len(cts) - 1 - i) * rh
            col = S.paper_palette_organoid_celltypes.get(c, "#888888")
            ax.fill_between(xr, y0, y0 + dens, color=col, alpha=0.8, zorder=i + 2, linewidth=0)
            ax.plot(xr, y0 + dens, color=col, lw=0.5, zorder=i + 3)
            ax.axhline(y0, color="0.9", lw=0.3, zorder=1)
            med = np.median(cd)
            mh = (kde([med])[0] / kde(xr).max()) * rh * 0.92
            ax.plot(
                [med, med],
                [y0, y0 + mh],
                color="white",
                lw=0.7,
                ls="--",
                zorder=i + 4,
                alpha=0.9,
            )
            ax.text(
                -0.11,
                y0 + rh * 0.35,
                c,
                fontsize=lblf,
                va="center",
                ha="right",
                color=col,
                fontweight="bold",
            )
        ax.annotate(
            "",
            xy=(1.08, -0.7),
            xytext=(-0.08, -0.7),
            arrowprops=dict(arrowstyle="-|>", color="dimgray", lw=0.8, mutation_scale=6),
        )
        ax.text(
            0.5,
            -1.05,
            ("differentiation" if half else "differentiation (proportion principal curve)"),
            fontsize=lblf,
            ha="center",
            color="dimgray",
            style="italic",
        )
        ax.set_title(
            L.split()[0][:3].upper() if half else L,
            fontsize=S.FONT_LABEL,
            color=lcol,
            fontweight="bold",
            pad=2,
        )
        ax.set_yticks([])
        ax.set_xticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
    if half:
        fig.subplots_adjust(left=0.11, right=0.99, top=0.90, bottom=0.10, wspace=0.55)
    else:
        fig.subplots_adjust(left=0.055, right=0.99, top=0.92, bottom=0.10, wspace=0.20)
    return fig


S.save_and_show(
    _build_s3a((S.FULL_W_IN, S.QUARTER_H_IN), False),
    "fig_s3a_differentiation_density__quarter_full",
    figure=FIG,
)
S.save_and_show(
    _build_s3a((S.HALF_W_IN, S.QUARTER_H_IN), True),
    "fig_s3a_differentiation_density__quarter_half",
    figure=FIG,
)
# S3b: y = nominal -log10 p; colour by adjusted-p significance;
# no horizontal padj line (not meaningful on a nominal-p axis) -> matches the main fig2c volcano
# (full y-range shown, as in fig2c; no clipping needed). Faint dpsi=0 line only.
for _lin_, _ in [("excitatory", "x"), ("inhibitory", "x")]:
    _dd = comp[_lin_]
    _dd["nl_nom"] = -np.log10(_dd["p_value_chisq"].clip(lower=1e-300))


def _build_s3b(figsize, half, sigcol="sig"):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    nlab = 3
    LINS_ = [("excitatory", "Excitatory neurons"), ("inhibitory", "Inhibitory neurons")]
    for ax, (lin_, linlabel) in zip(axes, LINS_):
        d = comp[lin_]
        y = d["nl_nom"].values
        cs = d["composite_score_abs"].values
        size = 1 + 9 * (cs - np.nanmin(cs)) / (np.nanmax(cs) - np.nanmin(cs) + 1e-9)
        for cls, col, z in [
            ("NS", S.COL_NS, 1),
            ("short", S.COL_SHORT, 2),
            ("long", S.COL_LONG, 2),
        ]:
            m = (d[sigcol] == cls).values
            ax.scatter(
                d.loc[m, "delta_psi"],
                y[m],
                c=col,
                s=size[m],
                alpha=0.35 if cls == "NS" else 0.7,
                edgecolors="none",
                rasterized=True,
                zorder=z,
            )
        ax.axvline(0, color="0.5", ls="-", lw=0.4, alpha=0.5, zorder=0)
        ax.set_ylim(top=ax.get_ylim()[1] * 1.20)
        a = asd[asd["lineage"] == lin_]
        if len(a):
            am = d["gene_name"].isin(a["gene_name"]).values
            ax.scatter(
                d.loc[am, "delta_psi"],
                y[am],
                s=14,
                marker="*",
                facecolors="none",
                edgecolors=ASD_STAR,
                linewidths=0.5,
                zorder=5,
            )
        top = d[d[sigcol] != "NS"].nlargest(nlab, "composite_score_abs")
        texts = [
            ax.text(
                r["delta_psi"],
                yi,
                r["gene_name"],
                fontsize=S.FONT_ANNOT - 1,
                fontstyle="italic",
                zorder=10,
            )
            for (_, r), yi in zip(top.iterrows(), y[top.index.values])
        ]
        if HAS_AT and texts:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.4))
        ns = int((d[sigcol] == "short").sum())
        nl_ = int((d[sigcol] == "long").sum())
        ax.text(
            0.02,
            0.97,
            f"{ns:,}",
            transform=ax.transAxes,
            color=S.COL_SHORT,
            fontsize=S.FONT_ANNOT - 0.5,
            fontweight="bold",
            va="top",
            ha="left",
        )
        ax.text(
            0.98,
            0.97,
            f"{nl_:,}",
            transform=ax.transAxes,
            color=S.COL_LONG,
            fontsize=S.FONT_ANNOT - 0.5,
            fontweight="bold",
            va="top",
            ha="right",
        )
        ax.set_title(
            linlabel.split()[0][:3].upper() if half else linlabel,
            fontsize=S.FONT_ANNOT if half else S.FONT_LABEL,
            color=S.paper_palette_lineage[lin_],
            fontweight="bold",
            pad=2,
        )
        ax.set_xlabel(r"$\Delta\psi$ (pseudotime)", fontsize=S.FONT_LABEL)
        if ax is axes[0]:
            ax.set_ylabel(r"$-\log_{10}p$ (nominal)", fontsize=S.FONT_LABEL)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.tick_params(labelsize=S.FONT_TICK)
        ax.spines[["top", "right"]].set_visible(False)
    leg = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_SHORT,
            markersize=3.5,
            label="shortening",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_LONG,
            markersize=3.5,
            label="lengthening",
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="w",
            markerfacecolor="none",
            markeredgecolor=ASD_STAR,
            markersize=5,
            label="ASD-risk",
        ),
    ]
    axes[0].legend(
        handles=leg,
        fontsize=S.FONT_LEGEND - 0.5,
        frameon=False,
        loc="upper center",
        ncol=3,
        bbox_to_anchor=(1.05, 1.16),
        handletextpad=0.2,
        columnspacing=0.6,
    )
    fig.subplots_adjust(
        left=0.07, right=0.98, top=0.83, bottom=0.16, wspace=0.16 if not half else 0.22
    )
    return fig


S.save_and_show(
    _build_s3b((S.FULL_W_IN, S.QUARTER_H_IN), False),
    "fig_s3b_pseudotime_dpsi_volcano__quarter_full",
    figure=FIG,
)
S.save_and_show(
    _build_s3b((S.HALF_W_IN, S.QUARTER_H_IN), True),
    "fig_s3b_pseudotime_dpsi_volcano__quarter_half",
    figure=FIG,
)

### S3a variant: 'APA-based pseudotime' x-label, aligned + closer to axis


In [ ]:
# --- S3a APA-label variant: x-axis "APA-based pseudotime", labels closer to the axis and
# vertically aligned across EXC/INH (common y-limit so the label sits at the same height). ---
def _build_s3a_apa(figsize):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    lblf = S.FONT_TICK - 1
    # common top across lineages so the arrow + x-label align vertically in both subpanels
    ncts = []
    for L, order, _lc in _LINS_S3A:
        lm = _lin3 == L
        ncts.append(len([c for c in order if (_ct3[lm] == c).any()]))
    top = max(ncts)
    for ax, (L, order, lcol) in zip(axes, _LINS_S3A):
        lm = _lin3 == L
        sub_um = _um3[lm]
        sub_ct = _ct3[lm]
        cts = [c for c in order if (sub_ct == c).any()]
        axis = principal_curve(sub_um.astype(float))
        means = [axis[sub_ct == c].mean() for c in cts]
        if spearmanr(range(len(cts)), means)[0] < 0:
            axis = -axis
        axis = (axis - axis.min()) / (axis.max() - axis.min() + 1e-9)
        xr = np.linspace(-0.08, 1.08, 400)
        rh = 1.0
        for i, c in enumerate(cts):
            cd = axis[sub_ct == c]
            if len(cd) < 5:
                continue
            kde = gaussian_kde(cd, bw_method=0.35)
            dens = kde(xr)
            dens = dens / dens.max() * rh * 0.92
            y0 = (len(cts) - 1 - i) * rh
            col = S.paper_palette_organoid_celltypes.get(c, "#888888")
            ax.fill_between(xr, y0, y0 + dens, color=col, alpha=0.8, zorder=i + 2, linewidth=0)
            ax.plot(xr, y0 + dens, color=col, lw=0.5, zorder=i + 3)
            ax.axhline(y0, color="0.9", lw=0.3, zorder=1)
            med = np.median(cd)
            mh = (kde([med])[0] / kde(xr).max()) * rh * 0.92
            ax.plot(
                [med, med],
                [y0, y0 + mh],
                color="white",
                lw=0.7,
                ls="--",
                zorder=i + 4,
                alpha=0.9,
            )
            ax.text(
                -0.11,
                y0 + rh * 0.35,
                c,
                fontsize=lblf,
                va="center",
                ha="right",
                color=col,
                fontweight="bold",
            )
        # arrow (pseudotime axis) + label, aligned across panels via the common ylim below
        ax.annotate(
            "",
            xy=(1.08, -0.55),
            xytext=(-0.08, -0.55),
            arrowprops=dict(arrowstyle="-|>", color="dimgray", lw=0.8, mutation_scale=6),
        )
        ax.text(
            0.5,
            -0.88,
            "APA-based pseudotime",
            fontsize=lblf,
            ha="center",
            color="dimgray",
            style="italic",
        )
        ax.set_title(
            L.split()[0][:3].upper(),
            fontsize=S.FONT_LABEL,
            color=lcol,
            fontweight="bold",
            pad=2,
        )
        ax.set_yticks([])
        ax.set_xticks([])
        ax.set_ylim(-1.15, top + 0.05)  # Common ylim -> arrow + x-label at identical height in both
        for sp in ax.spines.values():
            sp.set_visible(False)
    fig.subplots_adjust(left=0.11, right=0.99, top=0.90, bottom=0.10, wspace=0.55)
    return fig


S.save_and_show(
    _build_s3a_apa((S.HALF_W_IN, S.QUARTER_H_IN)),
    "fig_s3a_differentiation_density__quarter_half_apalabel",
    figure=FIG,
)

### S3b variant: coloured by permutation-adjusted significance (delta-psi volcano)


In [ ]:
# --- S3b perm-FDR variant: colour dots + counts by permutation-adjusted significance
# (padj_perm<0.05), split by delta_psi sign, no effect-size gate -> the delta-psi companion to
# the fig2c rho volcano's perm colouring (which additionally gates on |rho_W|>0.2). ---
for _lin, _ in LINS:
    _d = comp[_lin]
    if "padj_perm" not in _d.columns:
        _permf = (
            f"{KB}/dirmult_results/{_lin}/permutation_batch_number/"
            f"dirmult_regression_permutation_optimized2_joint_df5_pt-dpt_pseudotime_{_lin}_cov-batch_number.csv"
        )
        _pm = pd.read_csv(_permf)[["utr_name", "p_value_perm"]]
        _d = _d.merge(_pm, left_on="gene_id", right_on="utr_name", how="left")
        _d["padj_perm"] = S.benjamini_hochberg(_d["p_value_perm"].values)
    _d["sig_perm"] = np.where(
        (_d["padj_perm"] < PADJ_THR) & (_d["delta_psi"] < 0),
        "short",
        np.where((_d["padj_perm"] < PADJ_THR) & (_d["delta_psi"] > 0), "long", "NS"),
    )
    comp[_lin] = _d
    print(
        f"{_lin}: perm-FDR (delta_psi sign) short={int((_d.sig_perm == 'short').sum())} long={int((_d.sig_perm == 'long').sum())}"
    )

S.save_and_show(
    _build_s3b((S.HALF_W_IN, S.QUARTER_H_IN), True, sigcol="sig_perm"),
    "fig_s3b_pseudotime_dpsi_volcano__quarter_half_perm",
    figure=FIG,
)

### S3b variant: coloured by chi-sq ∩ permutation intersection (conservative DTU set)


In [ ]:
# --- S3b chisq∩perm intersection variant: colour by genes significant by both the chi-sq LRT
# (padj<0.05) and permutation (padj_perm<0.05), split by delta_psi sign. The conservative DTU set
# (vs the perm-alone __perm and chisq-alone base s3b). Reuses padj_perm added in the perm cell. ---
for _lin, _ in LINS:
    _d = comp[_lin]
    _d["sig_int"] = np.where(
        (_d["padj"] < PADJ_THR) & (_d["padj_perm"] < PADJ_THR) & (_d["delta_psi"] < 0),
        "short",
        np.where(
            (_d["padj"] < PADJ_THR) & (_d["padj_perm"] < PADJ_THR) & (_d["delta_psi"] > 0),
            "long",
            "NS",
        ),
    )
    comp[_lin] = _d
    print(
        f"{_lin}: chisq&perm short={int((_d.sig_int == 'short').sum())} long={int((_d.sig_int == 'long').sum())}"
    )
S.save_and_show(
    _build_s3b((S.HALF_W_IN, S.QUARTER_H_IN), True, sigcol="sig_int"),
    "fig_s3b_pseudotime_dpsi_volcano__quarter_half_chisqperm",
    figure=FIG,
)